In [ ]:
"""
Plant Disease Detection using CNNs - Google Colab Version
Comparing Custom CNN vs Transfer Learning Models
With Auto-Download and Interactive Image Upload
"""

# ============================================
# INSTALLATION
# ============================================

print("🔧 Installing required packages...")
print("This may take a few minutes...\n")

# Install TensorFlow
!pip install -q tensorflow

# Install other required packages
!pip install -q scikit-learn matplotlib seaborn pillow

print("✓ All packages installed successfully!\n")

# ============================================
# IMPORTS
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from IPython.display import display, Image as IPImage, HTML
import io
from PIL import Image
import os
import zipfile
import urllib.request

# Import files from google.colab
try:
    from google.colab import files as colab_files
except ImportError:
    print("Warning: Not running in Google Colab environment")
    colab_files = None

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✓ All libraries imported successfully!")
print(f"TensorFlow version: {tf._version_}\n")

# ============================================
# 1. AUTO-DOWNLOAD DATASET FROM KAGGLE
# ============================================

def download_dataset_from_kaggle():
    """Download Plant Village dataset from Kaggle automatically"""
    print("="*70)
    print("🌐 AUTO-DOWNLOADING PLANTVILLAGE DATASET FROM KAGGLE")
    print("="*70)
    print("\nThis dataset contains 54,305 images across 38 plant disease classes")
    print("Download size: ~800 MB (may take 3-5 minutes)\n")

    # Install kaggle package
    print("📦 Installing Kaggle API...")
    !pip install -q kaggle

    print("\n✓ Kaggle API installed!")

    # Upload kaggle.json
    print("\n" + "="*70)
    print("🔑 KAGGLE API AUTHENTICATION")
    print("="*70)
    print("\nYou need a Kaggle API token to download the dataset.")
    print("\n📝 How to get your kaggle.json file:")
    print("   1. Go to: https://www.kaggle.com/settings")
    print("   2. Scroll to 'API' section")
    print("   3. Click 'Create New API Token'")
    print("   4. Download the kaggle.json file")
    print("\n👇 Upload your kaggle.json file below:\n")

    uploaded = colab_files.upload()

    if 'kaggle.json' not in uploaded:
        print("❌ Error: kaggle.json not found. Please upload the correct file.")
        return None

    # Setup Kaggle credentials
    print("\n🔐 Setting up credentials...")
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("✓ Credentials configured!")

    # Download dataset
    print("\n" + "="*70)
    print("⬇  DOWNLOADING DATASET")
    print("="*70)
    print("\nDownloading PlantVillage dataset...")
    print("Please wait, this may take 3-5 minutes...\n")

    !kaggle datasets download -d arjuntejaswi/plant-village

    # Extract dataset
    print("\n📦 Extracting dataset...")
    with zipfile.ZipFile('plant-village.zip', 'r') as zip_ref:
        zip_ref.extractall('dataset')

    print("✓ Extraction complete!")

    # Find the data directory
    data_dir = 'dataset'
    possible_paths = [
        'dataset/PlantVillage',
        'dataset/color',
        'dataset/Plant_leave_diseases_dataset_without_augmentation'
    ]

    for path in possible_paths:
        if os.path.exists(path):
            data_dir = path
            break

    # If not found in common paths, search for it
    if data_dir == 'dataset':
        for root, dirs, files in os.walk('dataset'):
            if len(dirs) > 10 and any(f.endswith(('.jpg', '.png', '.jpeg')) for f in os.listdir(os.path.join(root, dirs[0])) if os.path.isdir(os.path.join(root, dirs[0]))):
                data_dir = root
                break

    print(f"\n✓ Dataset located at: {data_dir}")

    # Show dataset info
    try:
        classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
        print(f"✓ Found {len(classes)} disease classes")

        total_images = sum([len(os.listdir(os.path.join(data_dir, c))) for c in classes])
        print(f"✓ Total images: {total_images}")

        print("\n📋 Sample classes:")
        for i, cls in enumerate(classes[:5]):
            img_count = len(os.listdir(os.path.join(data_dir, cls)))
            print(f"   - {cls}: {img_count} images")
        if len(classes) > 5:
            print(f"   ... and {len(classes) - 5} more classes")
    except Exception as e:
        print(f"Note: Could not read class information: {e}")

    print("\n" + "="*70)
    print("✅ DATASET READY FOR TRAINING!")
    print("="*70 + "\n")

    return data_dir

# ============================================
# 2. DATA LOADING AND PREPROCESSING
# ============================================

class DataLoader:
    def _init_(self, data_dir, img_size=(128, 128), batch_size=32):
        self.data_dir = data_dir
        self.img_size = img_size
        self.batch_size = batch_size

    def create_generators(self):
        """Create data generators with augmentation"""
        train_datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            shear_range=0.2,
            zoom_range=0.2,
            horizontal_flip=True,
            fill_mode='nearest',
            validation_split=0.2
        )

        test_datagen = ImageDataGenerator(rescale=1./255)

        train_generator = train_datagen.flow_from_directory(
            self.data_dir,
            target_size=self.img_size,
            batch_size=self.batch_size,
            class_mode='categorical',
            subset='training'
        )

        val_generator = train_datagen.flow_from_directory(
            self.data_dir,
            target_size=self.img_size,
            batch_size=self.batch_size,
            class_mode='categorical',
            subset='validation'
        )

        return train_generator, val_generator

# ============================================
# 3. CUSTOM CNN MODEL
# ============================================

def create_custom_cnn(input_shape, num_classes):
    """Create a custom CNN architecture"""
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape, padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 4
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Classifier
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

    return model

# ============================================
# 4. TRANSFER LEARNING MODELS
# ============================================

def create_transfer_learning_model(base_model_name, input_shape, num_classes, trainable_layers=0):
    """Create transfer learning model with different architectures"""

    # Load base model
    if base_model_name == 'VGG16':
        base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    elif base_model_name == 'ResNet50':
        base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    elif base_model_name == 'MobileNetV2':
        base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    else:
        raise ValueError("Model not supported")

    # Freeze base model layers
    base_model.trainable = False

    # If trainable_layers > 0, unfreeze last few layers for fine-tuning
    if trainable_layers > 0:
        base_model.trainable = True
        for layer in base_model.layers[:-trainable_layers]:
            layer.trainable = False

    # Create model
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

    return model

# ============================================
# 5. TRAINING FUNCTION
# ============================================

def train_model(model, train_gen, val_gen, model_name, epochs=50):
    """Train the model with callbacks"""

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Callbacks
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)

    # Train
    print(f"\n{'='*50}")
    print(f"🚀 Training {model_name}")
    print(f"{'='*50}\n")

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    return history

# ============================================
# 6. EVALUATION AND VISUALIZATION
# ============================================

def plot_training_history(histories, model_names):
    """Plot training and validation accuracy/loss"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Accuracy
    for history, name in zip(histories, model_names):
        axes[0, 0].plot(history.history['accuracy'], label=f'{name} - Train')
        axes[0, 0].plot(history.history['val_accuracy'], label=f'{name} - Val', linestyle='--')
    axes[0, 0].set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Loss
    for history, name in zip(histories, model_names):
        axes[0, 1].plot(history.history['loss'], label=f'{name} - Train')
        axes[0, 1].plot(history.history['val_loss'], label=f'{name} - Val', linestyle='--')
    axes[0, 1].set_title('Model Loss Comparison', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Final accuracy comparison
    final_train_acc = [h.history['accuracy'][-1] for h in histories]
    final_val_acc = [h.history['val_accuracy'][-1] for h in histories]
    x = np.arange(len(model_names))
    width = 0.35
    axes[1, 0].bar(x - width/2, final_train_acc, width, label='Train Accuracy', color='skyblue')
    axes[1, 0].bar(x + width/2, final_val_acc, width, label='Val Accuracy', color='orange')
    axes[1, 0].set_title('Final Accuracy Comparison', fontsize=14, fontweight='bold')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(model_names, rotation=45, ha='right')
    axes[1, 0].legend()
    axes[1, 0].grid(True, axis='y')

    # Training time (epochs to converge)
    epochs_trained = [len(h.history['accuracy']) for h in histories]
    axes[1, 1].bar(model_names, epochs_trained, color='lightgreen')
    axes[1, 1].set_title('Epochs to Convergence', fontsize=14, fontweight='bold')
    axes[1, 1].set_ylabel('Number of Epochs')
    axes[1, 1].set_xticklabels(model_names, rotation=45, ha='right')
    axes[1, 1].grid(True, axis='y')

    plt.tight_layout()
    plt.savefig('training_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

def evaluate_model(model, val_gen, model_name):
    """Evaluate model and print metrics"""
    print(f"\n{'='*50}")
    print(f"📊 Evaluating {model_name}")
    print(f"{'='*50}\n")

    # Get predictions
    val_gen.reset()
    predictions = model.predict(val_gen, verbose=1)
    y_pred = np.argmax(predictions, axis=1)
    y_true = val_gen.classes

    # Classification report
    class_names = list(val_gen.class_indices.keys())
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
    plt.title(f'Confusion Matrix - {model_name}', fontsize=14, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{model_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

    return accuracy_score(y_true, y_pred)

def create_results_table(results):
    """Create comparison table of all models"""
    df = pd.DataFrame(results)
    print("\n" + "="*90)
    print("🏆 FINAL RESULTS COMPARISON")
    print("="*90)
    print(df.to_string(index=False))
    print("="*90)

    # Save to CSV
    df.to_csv('model_comparison_results.csv', index=False)

    # Visualize results
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Accuracy comparison
    colors = plt.cm.viridis(np.linspace(0, 1, len(df)))
    axes[0].barh(df['Model'], df['Validation Accuracy'], color=colors)
    axes[0].set_xlabel('Accuracy', fontweight='bold')
    axes[0].set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
    axes[0].grid(True, axis='x', alpha=0.3)

    # Parameters comparison
    axes[1].barh(df['Model'], df['Total Parameters'] / 1e6, color=colors)
    axes[1].set_xlabel('Parameters (Millions)', fontweight='bold')
    axes[1].set_title('Model Size Comparison', fontsize=14, fontweight='bold')
    axes[1].grid(True, axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig('final_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

# ============================================
# 7. IMAGE UPLOAD AND PREDICTION INTERFACE
# ============================================

class PlantDiseasePredictor:
    def _init_(self, models_dict, class_names, img_size=(128, 128)):
        """
        Initialize predictor with trained models
        models_dict: dictionary of {model_name: model}
        class_names: list of class names
        """
        self.models = models_dict
        self.class_names = class_names
        self.img_size = img_size

    def preprocess_image(self, image_path):
        """Preprocess uploaded image"""
        img = load_img(image_path, target_size=self.img_size)
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = img_array / 255.0
        return img, img_array

    def predict_all_models(self, image_path):
        """Get predictions from all models"""
        print(f"\n{'='*70}")
        print("🔍 ANALYZING IMAGE WITH ALL MODELS")
        print(f"{'='*70}\n")

        img, img_array = self.preprocess_image(image_path)

        # Display image
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Uploaded Image', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        results = []
        all_predictions = {}

        # Get predictions from each model
        for model_name, model in self.models.items():
            predictions = model.predict(img_array, verbose=0)
            predicted_class_idx = np.argmax(predictions[0])
            predicted_class = self.class_names[predicted_class_idx]
            confidence = predictions[0][predicted_class_idx] * 100

            results.append({
                'Model': model_name,
                'Predicted Class': predicted_class,
                'Confidence': f'{confidence:.2f}%'
            })

            all_predictions[model_name] = predictions[0]

            print(f"✓ {model_name:20s} -> {predicted_class:40s} (Confidence: {confidence:6.2f}%)")

        # Create results dataframe
        df_results = pd.DataFrame(results)

        # Visualize predictions
        self.visualize_predictions(all_predictions, img)

        return df_results

    def visualize_predictions(self, all_predictions, original_img):
        """Visualize predictions from all models"""
        n_models = len(all_predictions)
        fig, axes = plt.subplots(1, n_models + 1, figsize=(5 * (n_models + 1), 5))

        # Show original image
        axes[0].imshow(original_img)
        axes[0].set_title('Original Image', fontweight='bold', fontsize=12)
        axes[0].axis('off')

        # Show predictions for each model
        for idx, (model_name, predictions) in enumerate(all_predictions.items(), 1):
            top_5_idx = np.argsort(predictions)[-5:][::-1]
            top_5_classes = [self.class_names[i] for i in top_5_idx]
            top_5_probs = predictions[top_5_idx] * 100

            colors = plt.cm.viridis(np.linspace(0, 1, 5))
            axes[idx].barh(top_5_classes, top_5_probs, color=colors)
            axes[idx].set_xlabel('Confidence (%)', fontweight='bold')
            axes[idx].set_title(f'{model_name}\nTop 5 Predictions', fontweight='bold', fontsize=12)
            axes[idx].grid(True, axis='x', alpha=0.3)

        plt.tight_layout()
        plt.show()

    def upload_and_predict(self):
        """Interactive function to upload and predict"""
        print("\n" + "="*70)
        print("📤 UPLOAD AN IMAGE FOR DISEASE DETECTION")
        print("="*70)
        print("\nSupported formats: JPG, JPEG, PNG")
        print("Click 'Choose Files' button below to upload an image\n")

        uploaded = colab_files.upload()

        for filename in uploaded.keys():
            print(f"\n✓ Uploaded: {filename}")

            # Save file temporarily
            with open(filename, 'wb') as f:
                f.write(uploaded[filename])

            # Get predictions
            results_df = self.predict_all_models(filename)

            # Display results table
            print("\n" + "="*70)
            print("📋 PREDICTION RESULTS")
            print("="*70)
            display(HTML(results_df.to_html(index=False)))

            # Clean up
            if os.path.exists(filename):
                os.remove(filename)

# ============================================
# 8. MAIN EXECUTION
# ============================================

def main():
    """Main execution function for Google Colab"""

    print("\n" + "="*70)
    print("🌿 PLANT DISEASE DETECTION USING CNNs")
    print("="*70)
    print("\nThis notebook will:")
    print("1. 🌐 Auto-download PlantVillage dataset from Kaggle")
    print("2. 🤖 Train 4 different CNN models (Custom + 3 Transfer Learning)")
    print("3. 📊 Compare model performances")
    print("4. 📤 Provide interactive image upload for testing")
    print("\n" + "="*70 + "\n")

    # Configuration
    IMG_SIZE = (128, 128)
    BATCH_SIZE = 32
    EPOCHS = 50

    # Step 1: Download Dataset
    data_dir = download_dataset_from_kaggle()

    if data_dir is None:
        print("❌ Dataset download failed. Please check your Kaggle credentials.")
        return None

    # Step 2: Load data
    print("\n" + "="*70)
    print("STEP 2: LOADING AND PREPARING DATA")
    print("="*70)

    loader = DataLoader(data_dir, IMG_SIZE, BATCH_SIZE)
    train_gen, val_gen = loader.create_generators()

    num_classes = len(train_gen.class_indices)
    input_shape = (*IMG_SIZE, 3)
    class_names = list(train_gen.class_indices.keys())

    print(f"\n✓ Number of classes: {num_classes}")
    print(f"✓ Training samples: {train_gen.samples}")
    print(f"✓ Validation samples: {val_gen.samples}")
    print(f"\n📋 Classes: {class_names[:5]}{'...' if len(class_names) > 5 else ''}")

    # Step 3: Train models
    print("\n" + "="*70)
    print("STEP 3: TRAINING MODELS")
    print("="*70)
    print("\nWe will train 4 different models:")
    print("1. Custom CNN (built from scratch)")
    print("2. VGG16 (Transfer Learning)")
    print("3. ResNet50 (Transfer Learning)")
    print("4. MobileNetV2 (Transfer Learning)")

    models_to_train = [
        ('Custom CNN', lambda: create_custom_cnn(input_shape, num_classes)),
        ('VGG16', lambda: create_transfer_learning_model('VGG16', input_shape, num_classes)),
        ('ResNet50', lambda: create_transfer_learning_model('ResNet50', input_shape, num_classes)),
        ('MobileNetV2', lambda: create_transfer_learning_model('MobileNetV2', input_shape, num_classes))
    ]

    histories = []
    trained_models = {}
    results = []

    for model_name, model_func in models_to_train:
        print(f"\n{'#'*70}")
        print(f"# {model_name}")
        print(f"{'#'*70}")

        model = model_func()
        print("\nModel Architecture:")
        model.summary()

        history = train_model(model, train_gen, val_gen, model_name, EPOCHS)
        histories.append(history)
        trained_models[model_name] = model

        val_accuracy = evaluate_model(model, val_gen, model_name)

        results.append({
            'Model': model_name,
            'Training Accuracy': f"{history.history['accuracy'][-1]:.4f}",
            'Validation Accuracy': f"{val_accuracy:.4f}",
            'Total Parameters': model.count_params(),
            'Trainable Parameters': sum([tf.size(w).numpy() for w in model.trainable_weights])
        })

        model.save(f'{model_name.replace(" ", "_")}_model.h5')
        print(f"\n✓ Model saved as {model_name.replace(' ', '_')}_model.h5")

    # Step 4: Compare results
    print("\n" + "="*70)
    print("STEP 4: MODEL COMPARISON")
    print("="*70)

    model_names = [name for name, _ in models_to_train]
    plot_training_history(histories, model_names)
    create_results_table(results)

    # Step 5: Interactive testing
    print("\n" + "="*70)
    print("STEP 5: INTERACTIVE TESTING")
    print("="*70)

    predictor = PlantDiseasePredictor(trained_models, class_names, IMG_SIZE)

    print("\n✅ ALL MODELS TRAINED SUCCESSFULLY!")
    print("\n" + "="*70)
    print("📤 READY FOR IMAGE TESTING!")
    print("="*70)
    print("\nYou can now test the models with new images.")
    print("\nRun the following command to upload and test an image:")
    print("\n>>> predictor.upload_and_predict()")
    print("\n" + "="*70)

    return predictor, trained_models, histories, results

# ============================================
# RUN THE MAIN FUNCTION
# ============================================

if _name_ == "_main_":
    predictor, trained_models, histories, results = main()